# Model Training v4 - Sample (Token) Level Training

This notebook removes the heavy speaker-level aggregation (`mean` / `std`). Instead, every utterance (token) is treated as an independent sample during training.

To prevent data leakage, we still use `StratifiedGroupKFold` split on `speaker_id`. This means if a speaker is in the test set, *all* of their utterances are in the test set, giving a true out-of-sample performance metric while preserving the underlying token distributions (e.g. distinguishing an 'a' from a 'u').

In [18]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, SelectFromModel
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sys.path.append('..')
from src.features import FeatureOptions, load_feature_tables

In [19]:
# Reproducibility & Config
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

SELECTED_TOKEN = None # Keep all tokens
MIN_SAMPLES_PER_CLASS = 50
MAX_SAMPLES_PER_CLASS = 500

BALANCE_HEALTHY_TO_PATHOLOGICAL = True
UPSAMPLE_HEALTHY_TO_PATHOLOGICAL = True

# Data quality guards
# Keep overlap filter OFF by default: it removes a large portion of pathological signal.
EXCLUDE_OVERLAP_SPEAKERS = False
EXCLUDE_MIXED_BINARY_SPEAKERS = True

N_SPLITS = 5
K_CANDIDATES = [50, 100, 150, 'auto']
THRESHOLD_GRID = np.linspace(0.35, 0.65, 11)

# Optimize for what you care about most
BINARY_THRESHOLD_OBJECTIVE = 'accuracy'  # 'accuracy' | 'balanced_accuracy'

TARGET_SOURCE_COL_PREFERENCE = 'pathology_de'
USE_GROUPED_TARGET = True
KEEP_UNMAPPED_LABELS = True
UNMAPPED_LABEL = 'Other'

DISEASE_GROUP_MAP = {
    'Morbus Parkinson': 'Neurological',
    'Rekurrensparese': 'Neurological',
    'Spasmodische Dysphonie': 'Neurological',
    'Phonationsknötchen': 'Structural',
    'Stimmlippenpolyp': 'Structural',
    'Reinke Ödem': 'Structural',
    'Laryngitis': 'Structural',
    'Hypotone Dysphonie': 'Functional',
    'Hyperfunktionelle Dysphonie': 'Functional',
    'Psychogene Dysphonie': 'Functional',
    'Funktionelle Dysphonie': 'Functional',
    'Phonationsknötchen': 'Structural',
    'Reinke Ödem': 'Structural',
}

opts = FeatureOptions(
    prefix=Path('..'),
    include_splits=True,
    random_seed=RANDOM_SEED,
    max_samples_per_class=MAX_SAMPLES_PER_CLASS,
    balance_healthy_to_pathological=BALANCE_HEALTHY_TO_PATHOLOGICAL,
    upsample_healthy_to_pathological=UPSAMPLE_HEALTHY_TO_PATHOLOGICAL,
    selected_token=SELECTED_TOKEN,
    num_workers=None,
)
opts

FeatureOptions(prefix=WindowsPath('..'), input_manifest=WindowsPath('data/processed/manifests/dataset_manifest.csv'), output_core=WindowsPath('data/processed/features/sample_core.csv'), output_acoustic=WindowsPath('data/processed/features/acoustic_features.csv'), output_multifractal=WindowsPath('data/processed/features/multifractal_features.csv'), output_opensmile=WindowsPath('data/processed/features/opensmile_features.csv'), output_splits=WindowsPath('data/processed/features/sample_splits.csv'), output_summary_json=WindowsPath('data/processed/features/feature_summary.json'), include_splits=True, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_seed=42, num_workers=None, max_samples_per_class=500, balance_healthy_to_pathological=True, upsample_healthy_to_pathological=True, normalize_audio=True, target_sample_rate=22050, selected_token=None, mfdfa_order=1, mfdfa_q_min=-5.0, mfdfa_q_max=5.0, mfdfa_q_step=1.0, mfdfa_num_scales=20)

In [20]:
# Load Features (Sample Level) with config-safety check
build_cfg_path = Path('..') / 'data' / 'processed' / 'features' / 'feature_build_config.json'
desired_cfg = {
    'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
    'balance_healthy_to_pathological': BALANCE_HEALTHY_TO_PATHOLOGICAL,
    'upsample_healthy_to_pathological': UPSAMPLE_HEALTHY_TO_PATHOLOGICAL,
    'target_sample_rate': opts.target_sample_rate,
    'selected_token': SELECTED_TOKEN,
}

force_rebuild = False
if build_cfg_path.exists():
    saved_cfg = json.loads(build_cfg_path.read_text(encoding='utf-8'))
    mismatches = {
        k: (saved_cfg.get(k), v)
        for k, v in desired_cfg.items()
        if saved_cfg.get(k) != v
    }
    if mismatches:
        force_rebuild = True
        print('Config mismatch vs cached features -> forcing rebuild:')
        for k, (old_v, new_v) in mismatches.items():
            print(f'  {k}: cached={old_v} -> desired={new_v}')

if force_rebuild:
    feature_dir = Path('..') / 'data' / 'processed' / 'features'
    for p in [
        feature_dir / 'sample_core.csv',
        feature_dir / 'acoustic_features.csv',
        feature_dir / 'multifractal_features.csv',
        feature_dir / 'opensmile_features.csv',
        feature_dir / 'sample_splits.csv',
        feature_dir / 'feature_summary.json',
        feature_dir / 'feature_build_config.json',
    ]:
        if p.exists():
            p.unlink()
    print('Deleted stale cached feature artifacts. Rebuilding now...')

tables = load_feature_tables(options=opts, build_if_missing=True, save_if_built=True)

core_df = tables['core'].copy()
acoustic_df = tables['acoustic'].copy()
multifractal_df = tables['multifractal'].copy()
opensmile_df = tables.get('opensmile', pd.DataFrame()).copy()
splits_df = tables.get('splits', pd.DataFrame()).copy()

df = core_df.merge(acoustic_df, on='sample_key', how='left')
df = df.merge(multifractal_df, on='sample_key', how='left')
if not opensmile_df.empty:
    df = df.merge(opensmile_df, on='sample_key', how='left')
if not splits_df.empty:
    df = df.merge(splits_df, on='sample_key', how='left')

if 'feature_status' in df.columns:
    df = df[df['feature_status'].isin(['ok', 'partial_failure'])].copy()
if 'acoustic_status' in df.columns:
    df = df[df['acoustic_status'] == 'ok'].copy()
if 'mf_status' in df.columns:
    df = df[df['mf_status'] == 'ok'].copy()
if 'opensmile_status' in df.columns:
    df = df[df['opensmile_status'] == 'ok'].copy()

print('Merged sample-level shape:', df.shape)
print('Unique speakers:', df['speaker_id'].nunique())

g:\Projects\multifractal-speech-analysis\notebooks\..\src\features\feature_cache.py:75: DtypeWarning: Columns (0: acoustic_error) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


Merged sample-level shape: (8640, 212)
Unique speakers: 1331


In [21]:
# Build Targets
raw_target = df[TARGET_SOURCE_COL_PREFERENCE].astype(str).str.strip()
if USE_GROUPED_TARGET:
    mapped_target = raw_target.map(DISEASE_GROUP_MAP)
    mapped_target = mapped_target.fillna(raw_target if KEEP_UNMAPPED_LABELS else UNMAPPED_LABEL)
    df['target_label'] = mapped_target.astype(str)
else:
    df['target_label'] = raw_target

target_col = 'target_label'

# Optional quality guard 1: remove overlap speakers that appear in multiple pathologies
if EXCLUDE_OVERLAP_SPEAKERS:
    overlap_cols = [c for c in ['is_overlap_speaker', 'is_overlap_speaker_id'] if c in df.columns]
    if overlap_cols:
        overlap_mask = pd.Series(False, index=df.index)
        for c in overlap_cols:
            overlap_mask = overlap_mask | df[c].astype(str).str.lower().eq('true')
        removed = int(overlap_mask.sum())
        if removed > 0:
            print(f'Removing overlap-speaker rows: {removed}')
            df = df.loc[~overlap_mask].copy()

# Optional quality guard 2: remove speakers with mixed binary labels (healthy + pathological)
if EXCLUDE_MIXED_BINARY_SPEAKERS and 'speaker_id' in df.columns and 'is_healthy' in df.columns:
    grp = df.groupby(df['speaker_id'].astype(str))['is_healthy'].nunique()
    bad_speakers = set(grp[grp > 1].index.tolist())
    if bad_speakers:
        removed = int(df['speaker_id'].astype(str).isin(bad_speakers).sum())
        print(f'Removing mixed-label speaker rows: {removed} from {len(bad_speakers)} speakers')
        df = df.loc[~df['speaker_id'].astype(str).isin(bad_speakers)].copy()

small_classes = df[target_col].value_counts()[df[target_col].value_counts() < MIN_SAMPLES_PER_CLASS].index.tolist()
if small_classes:
    print(f'Dropping classes with < {MIN_SAMPLES_PER_CLASS} samples: {small_classes}')
    df = df[~df[target_col].isin(small_classes)].copy()

display(df[target_col].value_counts().to_frame('sample_count'))

Removing mixed-label speaker rows: 178 from 15 speakers


,sample_count
target_label,
healthy,4219
Structural,1717
Functional,1516
Neurological,1010


In [22]:
# Prep Training Matrices
meta_cols = {
    'sample_key', 'duplicate_class_key', 'recording_id', 'speaker_id', 'wav_path',
    'feature_status', 'feature_error', 'acoustic_status', 'acoustic_error',
    'mf_status', 'mf_error', 'opensmile_status', 'opensmile_error',
    'split', 'split_seed', 'pathology_de', 'pathology_en', target_col, 'is_healthy',
}

numeric_cols = [c for c in df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])]

# We add 'token' to the categorical columns so the model knows which phonetic sound it's looking at
categorical_cols = []
if 'sex' in df.columns:
    categorical_cols.append('sex')
if 'token' in df.columns:
    categorical_cols.append('token')

X = df[numeric_cols + categorical_cols].copy()
# Binary classification: 1=Healthy, 0=Pathological
y_bin = df['is_healthy'].astype(int).copy()
y_multi = df[target_col].astype(str).copy()
groups = df['speaker_id'].astype(str).copy()
tokens = df['token'].astype(str).copy() if 'token' in df.columns else pd.Series(['__missing__'] * len(df), index=df.index)

print('Total tokens/samples:', len(df))
print('Features per sample:', len(numeric_cols))
print('Categorical columns:', categorical_cols)
print('Unique tokens:', tokens.nunique())

Total tokens/samples: 8462
Features per sample: 191
Categorical columns: ['sex', 'token']
Unique tokens: 14


In [23]:
# Preprocessing Pipeline
# Important: Median imputation is dangerous for F0/MFDFA missing values (they represent unvoiced/unstructured noise, not "average" pitch).
# Instead of replacing NaN with median pitch (which disguises pathology as healthy), we impute with 0 
# and add missing indicators so tree models can split on "was this sound unvoiced/corrupted?"

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True)),
    ('scaler', PowerTransformer(method='yeo-johnson', standardize=True)),
    ('variance', VarianceThreshold(threshold=0.0)),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ],
    remainder='drop',
)

def make_pipe(model, k=50):
    if k == 'auto':
        selector = SelectFromModel(LinearSVC(penalty='l1', dual=False, C=0.05, class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED))
    else:
        selector = SelectKBest(f_classif, k=k)
        
    return Pipeline(steps=[
        ('prep', preprocessor),
        ('selector', selector),
        ('model', model),
    ])

In [24]:
def evaluate_binary_sample_cv(model, k, X, y, groups, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    
    fold_rows = []
    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        pipe = make_pipe(clone(model), k=k)
        pipe.fit(X_tr, y_tr)
        
        # Fast threshold tuning on train probas (objective configurable)
        p_tr = pipe.predict_proba(X_tr)[:, 1]
        best_thr, best_score = 0.5, -1.0
        for thr in THRESHOLD_GRID:
            pred_tr = (p_tr >= thr).astype(int)
            if BINARY_THRESHOLD_OBJECTIVE == 'accuracy':
                score = accuracy_score(y_tr, pred_tr)
            else:
                score = balanced_accuracy_score(y_tr, pred_tr)
            if score > best_score:
                best_score = score
                best_thr = float(thr)

        p_te = pipe.predict_proba(X_te)[:, 1]
        pred_te = (p_te >= best_thr).astype(int)

        fold_rows.append({
            'fold': fold,
            'threshold': best_thr,
            'accuracy': accuracy_score(y_te, pred_te),
            'balanced_accuracy': balanced_accuracy_score(y_te, pred_te),
            'f1_macro': f1_score(y_te, pred_te, average='macro', zero_division=0),
        })
        
    fold_df = pd.DataFrame(fold_rows)
    return fold_df, {
        'accuracy_mean': fold_df['accuracy'].mean(),
        'balanced_accuracy_mean': fold_df['balanced_accuracy'].mean(),
        'f1_macro_mean': fold_df['f1_macro'].mean(),
    }


def evaluate_binary_speaker_cv(model, k, X, y, groups, n_splits=5):
    """Train on token-level, evaluate on speaker-aggregated probabilities per fold."""
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    fold_rows = []

    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        g_tr, g_te = groups.iloc[tr], groups.iloc[te]

        pipe = make_pipe(clone(model), k=k)
        pipe.fit(X_tr, y_tr)

        # Tune threshold at speaker level on train split
        p_tr = pipe.predict_proba(X_tr)[:, 1]
        tr_eval = pd.DataFrame({
            'speaker_id': g_tr.astype(str).values,
            'y': y_tr.astype(int).values,
            'p': p_tr,
        })
        tr_spk = tr_eval.groupby('speaker_id', as_index=False).agg(y=('y', 'first'), p=('p', 'mean'))

        best_thr, best_score = 0.5, -1.0
        for thr in THRESHOLD_GRID:
            pred = (tr_spk['p'].values >= thr).astype(int)
            if BINARY_THRESHOLD_OBJECTIVE == 'accuracy':
                score = accuracy_score(tr_spk['y'].values, pred)
            else:
                score = balanced_accuracy_score(tr_spk['y'].values, pred)
            if score > best_score:
                best_score = score
                best_thr = float(thr)

        # Evaluate on speaker-aggregated test predictions
        p_te = pipe.predict_proba(X_te)[:, 1]
        te_eval = pd.DataFrame({
            'speaker_id': g_te.astype(str).values,
            'y': y_te.astype(int).values,
            'p': p_te,
        })
        te_spk = te_eval.groupby('speaker_id', as_index=False).agg(y=('y', 'first'), p=('p', 'mean'))
        pred_spk = (te_spk['p'].values >= best_thr).astype(int)

        fold_rows.append({
            'fold': fold,
            'threshold': best_thr,
            'accuracy': accuracy_score(te_spk['y'].values, pred_spk),
            'balanced_accuracy': balanced_accuracy_score(te_spk['y'].values, pred_spk),
            'f1_macro': f1_score(te_spk['y'].values, pred_spk, average='macro', zero_division=0),
        })

    fold_df = pd.DataFrame(fold_rows)
    return fold_df, {
        'accuracy_mean': fold_df['accuracy'].mean(),
        'balanced_accuracy_mean': fold_df['balanced_accuracy'].mean(),
        'f1_macro_mean': fold_df['f1_macro'].mean(),
    }


def evaluate_binary_speaker_token_specialist_cv(model, k, X, y, groups, tokens, n_splits=5, min_token_rows=40):
    """
    Train one global model + per-token specialist models within each fold, then
    aggregate probabilities at speaker level on test fold.
    """
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    fold_rows = []

    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr = X.iloc[tr].reset_index(drop=True)
        X_te = X.iloc[te].reset_index(drop=True)
        y_tr = y.iloc[tr].reset_index(drop=True)
        y_te = y.iloc[te].reset_index(drop=True)
        g_tr = groups.iloc[tr].astype(str).reset_index(drop=True)
        g_te = groups.iloc[te].astype(str).reset_index(drop=True)
        t_tr = tokens.iloc[tr].astype(str).reset_index(drop=True)
        t_te = tokens.iloc[te].astype(str).reset_index(drop=True)

        global_pipe = make_pipe(clone(model), k=k)
        global_pipe.fit(X_tr, y_tr)

        # Build token specialists only when enough rows and both classes exist
        token_pipes = {}
        for tok in sorted(t_tr.unique()):
            m = t_tr == tok
            if int(m.sum()) < min_token_rows:
                continue
            if y_tr[m].nunique() < 2:
                continue
            p = make_pipe(clone(model), k=k)
            p.fit(X_tr[m], y_tr[m])
            token_pipes[tok] = p

        # Threshold tuned on global train speaker probabilities
        p_tr = global_pipe.predict_proba(X_tr)[:, 1]
        tr_eval = pd.DataFrame({'speaker_id': g_tr.values, 'y': y_tr.values.astype(int), 'p': p_tr})
        tr_spk = tr_eval.groupby('speaker_id', as_index=False).agg(y=('y', 'first'), p=('p', 'mean'))

        best_thr, best_score = 0.5, -1.0
        for thr in THRESHOLD_GRID:
            pred = (tr_spk['p'].values >= thr).astype(int)
            if BINARY_THRESHOLD_OBJECTIVE == 'accuracy':
                score = accuracy_score(tr_spk['y'].values, pred)
            else:
                score = balanced_accuracy_score(tr_spk['y'].values, pred)
            if score > best_score:
                best_score = score
                best_thr = float(thr)

        # Token-specialist test probabilities with global fallback
        p_te = np.zeros(len(X_te), dtype=float)
        for tok in sorted(t_te.unique()):
            m = (t_te == tok).values
            local_pipe = token_pipes.get(tok, global_pipe)
            p_te[m] = local_pipe.predict_proba(X_te.iloc[m])[:, 1]

        te_eval = pd.DataFrame({'speaker_id': g_te.values, 'y': y_te.values.astype(int), 'p': p_te})
        te_spk = te_eval.groupby('speaker_id', as_index=False).agg(y=('y', 'first'), p=('p', 'mean'))
        pred_spk = (te_spk['p'].values >= best_thr).astype(int)

        fold_rows.append({
            'fold': fold,
            'threshold': best_thr,
            'accuracy': accuracy_score(te_spk['y'].values, pred_spk),
            'balanced_accuracy': balanced_accuracy_score(te_spk['y'].values, pred_spk),
            'f1_macro': f1_score(te_spk['y'].values, pred_spk, average='macro', zero_division=0),
            'n_token_specialists': len(token_pipes),
        })

    fold_df = pd.DataFrame(fold_rows)
    return fold_df, {
        'accuracy_mean': fold_df['accuracy'].mean(),
        'balanced_accuracy_mean': fold_df['balanced_accuracy'].mean(),
        'f1_macro_mean': fold_df['f1_macro'].mean(),
        'n_token_specialists_mean': fold_df['n_token_specialists'].mean(),
    }

In [ ]:
# Binary Classification Run
binary_models = {
    'LogReg': LogisticRegression(max_iter=3000, class_weight='balanced', C=1.0, random_state=RANDOM_SEED),
    'RandomForest': RandomForestClassifier(
        n_estimators=800, max_depth=None, min_samples_leaf=2,
        class_weight='balanced_subsample', random_state=RANDOM_SEED, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=800, learning_rate=0.03, num_leaves=63,
        min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=700, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=1.0, reg_lambda=2.0,
        random_state=RANDOM_SEED, n_jobs=-1, eval_metric='logloss'
    ),
}

# Token-level metrics
bin_rows = []
for k in K_CANDIDATES:
    for model_name, model in binary_models.items():
        fold_df, summ = evaluate_binary_sample_cv(model, k, X, y_bin, groups, n_splits=N_SPLITS)
        bin_rows.append({'model': model_name, 'k': k, **summ})

bin_results = pd.DataFrame(bin_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Token-level binary metrics')
display(bin_results)

# Speaker-level metrics (recommended for clinical decision use)
bin_spk_rows = []
for k in K_CANDIDATES:
    for model_name, model in binary_models.items():
        fold_df, summ = evaluate_binary_speaker_cv(model, k, X, y_bin, groups, n_splits=N_SPLITS)
        bin_spk_rows.append({'model': model_name, 'k': k, **summ})

bin_speaker_results = pd.DataFrame(bin_spk_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Speaker-level binary metrics (token probabilities aggregated by speaker)')
display(bin_speaker_results)

# Speaker-level token-specialist metrics (global + per-token experts)
specialist_rows = []
specialist_model = clone(binary_models['XGBoost'])
for k in ['auto', 150]:
    fold_df, summ = evaluate_binary_speaker_token_specialist_cv(
        specialist_model, k, X, y_bin, groups, tokens, n_splits=N_SPLITS, min_token_rows=40
    )
    specialist_rows.append({'model': 'XGBoost-TokenSpecialist', 'k': k, **summ})

specialist_results = pd.DataFrame(specialist_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Speaker-level binary metrics (token-specialist ensemble)')
display(specialist_results)

g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

Token-level binary metrics


,model,k,accuracy_mean,balanced_accuracy_mean,f1_macro_mean
0,XGBoost,auto,0.779728,0.779914,0.778105
1,LightGBM,auto,0.776777,0.777198,0.770404
2,XGBoost,150,0.774528,0.774721,0.772500
3,LightGBM,150,0.771339,0.771752,0.765291
4,LightGBM,100,0.765313,0.765739,0.758780
5,XGBoost,100,0.764720,0.764842,0.763377
6,RandomForest,150,0.763304,0.763827,0.754201
7,RandomForest,100,0.763304,0.763785,0.755424
8,LogReg,auto,0.762713,0.763007,0.759042
9,RandomForest,auto,0.759288,0.759825,0.748823


g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

In [ ]:
# Multi-Class Run (Pathological Only)
idx_patho = y_bin == 0
X_patho, y_patho, g_patho = X[idx_patho], y_multi[idx_patho], groups[idx_patho]
t_patho = tokens[idx_patho] if 'tokens' in globals() else pd.Series(['__missing__'] * len(X_patho), index=X_patho.index)

multi_models = {
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced_subsample', random_state=RANDOM_SEED, n_jobs=-1),
    'LightGBM': LGBMClassifier(n_estimators=300, learning_rate=0.05, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1, verbose=-1),
    'XGBoost': XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=1.0, reg_lambda=2.0,
        random_state=RANDOM_SEED, n_jobs=-1, eval_metric='mlogloss'
    ),
}

def _make_sample_weights(y_enc):
    classes = np.unique(y_enc)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_enc)
    weight_map = {cls: w for cls, w in zip(classes, weights)}
    return np.array([weight_map[v] for v in y_enc], dtype=float)

def evaluate_multiclass_sample_and_speaker_cv(model, k, X, y, groups, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    sample_rows, speaker_rows = [], []

    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        g_te = groups.iloc[te].astype(str)

        le = LabelEncoder()
        y_tr_enc = le.fit_transform(y_tr.astype(str))
        y_te_enc = le.transform(y_te.astype(str))
        sw = _make_sample_weights(y_tr_enc)

        pipe = make_pipe(clone(model), k=k)
        pipe.fit(X_tr, y_tr_enc, model__sample_weight=sw)
        pred_enc = pipe.predict(X_te)

        sample_rows.append({
            'fold': fold,
            'accuracy': accuracy_score(y_te_enc, pred_enc),
            'balanced_accuracy': balanced_accuracy_score(y_te_enc, pred_enc),
            'f1_macro': f1_score(y_te_enc, pred_enc, average='macro', zero_division=0),
        })

        # Speaker-level aggregation by averaging class probabilities
        proba = pipe.predict_proba(X_te)
        proba_df = pd.DataFrame(proba, columns=[f'c_{i}' for i in range(proba.shape[1])])
        proba_df['speaker_id'] = g_te.values
        spk_proba = proba_df.groupby('speaker_id', as_index=False).mean()

        spk_pred_enc = spk_proba[[c for c in spk_proba.columns if c.startswith('c_')]].values.argmax(axis=1)
        spk_true = pd.DataFrame({'speaker_id': g_te.values, 'y': y_te.astype(str).values}).groupby('speaker_id')['y'].agg(lambda s: s.mode().iloc[0]).reset_index()
        spk_true_enc = le.transform(spk_true['y'].values)

        speaker_rows.append({
            'fold': fold,
            'accuracy': accuracy_score(spk_true_enc, spk_pred_enc),
            'balanced_accuracy': balanced_accuracy_score(spk_true_enc, spk_pred_enc),
            'f1_macro': f1_score(spk_true_enc, spk_pred_enc, average='macro', zero_division=0),
        })

    sample_df = pd.DataFrame(sample_rows)
    speaker_df = pd.DataFrame(speaker_rows)

    sample_summary = {
        'accuracy_mean': sample_df['accuracy'].mean(),
        'balanced_accuracy_mean': sample_df['balanced_accuracy'].mean(),
        'f1_macro_mean': sample_df['f1_macro'].mean(),
    }
    speaker_summary = {
        'accuracy_mean': speaker_df['accuracy'].mean(),
        'balanced_accuracy_mean': speaker_df['balanced_accuracy'].mean(),
        'f1_macro_mean': speaker_df['f1_macro'].mean(),
    }
    return sample_summary, speaker_summary

def evaluate_multiclass_speaker_token_specialist_cv(model, k, X, y, groups, tokens, n_splits=5, min_token_rows=60):
    """
    Train one global multiclass model + per-token specialists and aggregate
    predictions at speaker level via majority vote.
    """
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    fold_rows = []

    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr = X.iloc[tr].reset_index(drop=True)
        X_te = X.iloc[te].reset_index(drop=True)
        y_tr = y.iloc[tr].astype(str).reset_index(drop=True)
        y_te = y.iloc[te].astype(str).reset_index(drop=True)
        g_te = groups.iloc[te].astype(str).reset_index(drop=True)
        t_tr = tokens.iloc[tr].astype(str).reset_index(drop=True)
        t_te = tokens.iloc[te].astype(str).reset_index(drop=True)

        le = LabelEncoder()
        y_tr_enc = le.fit_transform(y_tr)
        y_te_enc = le.transform(y_te)
        sw_global = _make_sample_weights(y_tr_enc)

        global_pipe = make_pipe(clone(model), k=k)
        global_pipe.fit(X_tr, y_tr_enc, model__sample_weight=sw_global)

        token_pipes = {}
        for tok in sorted(t_tr.unique()):
            m = t_tr == tok
            if int(m.sum()) < min_token_rows:
                continue
            y_tok = y_tr_enc[m]
            if pd.Series(y_tok).nunique() < 2:
                continue
            p = make_pipe(clone(model), k=k)
            sw_tok = _make_sample_weights(y_tok)
            p.fit(X_tr[m], y_tok, model__sample_weight=sw_tok)
            token_pipes[tok] = p

        # Row-level prediction using specialist if available
        pred_row = np.zeros(len(X_te), dtype=int)
        for tok in sorted(t_te.unique()):
            m = (t_te == tok).values
            local_pipe = token_pipes.get(tok, global_pipe)
            pred_row[m] = local_pipe.predict(X_te.iloc[m])

        # Token-level metrics
        row_acc = accuracy_score(y_te_enc, pred_row)
        row_ba = balanced_accuracy_score(y_te_enc, pred_row)
        row_f1 = f1_score(y_te_enc, pred_row, average='macro', zero_division=0)

        # Speaker-level majority vote
        te_eval = pd.DataFrame({
            'speaker_id': g_te.values,
            'y_true': y_te_enc,
            'y_pred': pred_row,
        })
        spk_pred = te_eval.groupby('speaker_id')['y_pred'].agg(lambda s: s.mode().iloc[0]).reset_index()
        spk_true = te_eval.groupby('speaker_id')['y_true'].agg(lambda s: s.mode().iloc[0]).reset_index()
        spk = spk_true.merge(spk_pred, on='speaker_id', how='inner')

        fold_rows.append({
            'fold': fold,
            'token_accuracy': row_acc,
            'token_balanced_accuracy': row_ba,
            'token_f1_macro': row_f1,
            'accuracy': accuracy_score(spk['y_true'].values, spk['y_pred'].values),
            'balanced_accuracy': balanced_accuracy_score(spk['y_true'].values, spk['y_pred'].values),
            'f1_macro': f1_score(spk['y_true'].values, spk['y_pred'].values, average='macro', zero_division=0),
            'n_token_specialists': len(token_pipes),
        })

    fold_df = pd.DataFrame(fold_rows)
    return {
        'accuracy_mean': fold_df['accuracy'].mean(),
        'balanced_accuracy_mean': fold_df['balanced_accuracy'].mean(),
        'f1_macro_mean': fold_df['f1_macro'].mean(),
        'token_accuracy_mean': fold_df['token_accuracy'].mean(),
        'n_token_specialists_mean': fold_df['n_token_specialists'].mean(),
    }

# Run baseline models
multi_sample_rows, multi_speaker_rows = [], []
for k in ['auto', 50, 100]:
    for model_name, model in multi_models.items():
        ssum, spksum = evaluate_multiclass_sample_and_speaker_cv(model, k, X_patho, y_patho, g_patho, n_splits=N_SPLITS)
        multi_sample_rows.append({'model': model_name, 'k': k, **ssum})
        multi_speaker_rows.append({'model': model_name, 'k': k, **spksum})

multi_results = pd.DataFrame(multi_sample_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Token-level multi-class metrics')
display(multi_results)

multi_speaker_results = pd.DataFrame(multi_speaker_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Speaker-level multi-class metrics (token probabilities aggregated by speaker)')
display(multi_speaker_results)

# Run token-specialist multi-class on best base model family
specialist_multi_rows = []
specialist_multi_model = clone(multi_models['XGBoost'])
for k in ['auto', 100]:
    summ = evaluate_multiclass_speaker_token_specialist_cv(
        specialist_multi_model, k, X_patho, y_patho, g_patho, t_patho, n_splits=N_SPLITS, min_token_rows=60
    )
    specialist_multi_rows.append({'model': 'XGBoost-TokenSpecialist', 'k': k, **summ})

specialist_multi_results = pd.DataFrame(specialist_multi_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Speaker-level multi-class metrics (token-specialist majority-vote ensemble)')
display(specialist_multi_results)

g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
g:\Projects\multifractal-speech-analysis\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
g:\Projects\multifractal-speec

Token-level multi-class metrics


,model,k,accuracy_mean,balanced_accuracy_mean,f1_macro_mean
0,XGBoost,100,0.510506,0.475140,0.469461
1,XGBoost,auto,0.505081,0.470814,0.466054
2,LightGBM,auto,0.504141,0.473170,0.470334
3,XGBoost,50,0.499185,0.465814,0.462005
4,RandomForest,auto,0.495891,0.460367,0.453379
5,LightGBM,50,0.492118,0.466765,0.465226
6,LightGBM,100,0.491892,0.461311,0.456877
7,RandomForest,100,0.491888,0.461725,0.457497
8,RandomForest,50,0.490232,0.463648,0.460445


Speaker-level multi-class metrics (token probabilities aggregated by speaker)


,model,k,accuracy_mean,balanced_accuracy_mean,f1_macro_mean
0,XGBoost,auto,0.566788,0.537101,0.524532
1,XGBoost,100,0.561723,0.535088,0.523057
2,XGBoost,50,0.556541,0.526767,0.513101
3,RandomForest,100,0.548653,0.521416,0.508921
4,LightGBM,auto,0.548147,0.520482,0.506673
5,LightGBM,100,0.545033,0.519497,0.507292
6,RandomForest,auto,0.542228,0.513219,0.498698
7,RandomForest,50,0.516760,0.496071,0.481271
8,LightGBM,50,0.513883,0.495927,0.479794
